# Joint Stress Supervision Ablation

This notebook evaluates **Zonal** models on a **validation-split evaluation** of `disc_dataset_edge_deriv_zonal.h5`.

It compares:
- `Edge_joint` = joint **Stress + LogLife** supervision
- `Edge_no_stress` = **LogLife-only** supervision

Comparable pair families:
- `ArGEnT_self_att_noSDF`
- `PointNetMLPJoint_FP`

`PointNetMLPJoint` has **no life-only checkpoint** under `Zonal/Edge_no_stress` and is reported as missing, then excluded from paired conclusions.


In [ ]:
from __future__ import annotations
import ast, hashlib, importlib.util, json, os, re, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
try:
    import torch
    import h5py
except ImportError as exc:
    raise RuntimeError('Install torch and h5py in the selected notebook kernel before executing this comparison.') from exc
CURRENT_DIR = Path.cwd()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / 'Uniform').exists() else CURRENT_DIR.parent
if not (REPO_ROOT / 'Uniform').exists(): raise RuntimeError(f'Repository root not found from {CURRENT_DIR}')
COMPARISON_DIR = REPO_ROOT / 'Comparison'
sys.path.insert(0, str(COMPARISON_DIR))
import eval_helpers as eh
SPLIT_SEED, EVAL_FRACTION = 42, 0.20
COMMIT = __import__('subprocess').check_output(['git','rev-parse','HEAD'], cwd=REPO_ROOT, text=True).strip()

RESULTS_DIR = COMPARISON_DIR / 'results' / '04_joint_stress_supervision'
FIGURES_DIR = RESULTS_DIR / 'figures'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

REGIME = 'Zonal'
DATASET_PATH = REPO_ROOT / 'Data_gen' / 'output' / 'disc_dataset_edge_deriv_zonal.h5'
JOINT_SOURCE = 'Edge'
LIFE_SOURCE = 'Edge_no_stress'
ABLATION_LABELS = {JOINT_SOURCE: 'Edge_joint', LIFE_SOURCE: 'Edge_no_stress'}
EXPECTED_FAMILIES = {
    JOINT_SOURCE: ['ArGEnT_self_att_noSDF', 'PointNetMLPJoint_FP', 'PointNetMLPJoint'],
    LIFE_SOURCE: ['ArGEnT_self_att_noSDF', 'PointNetMLPJoint_FP', 'PointNetMLPJoint'],
}
PAIR_FAMILIES = ['ArGEnT_self_att_noSDF', 'PointNetMLPJoint_FP']
EVALUATION_LABEL = 'validation-split evaluation'
warnings.filterwarnings('ignore', category=FutureWarning)
display(Markdown(f'**Commit:** `{COMMIT}`\n\n**Results directory:** `{RESULTS_DIR}`\n\n**Evaluation label:** **{EVALUATION_LABEL}**'))


## Checkpoint discovery and pair-compatibility inputs

The next cell discovers the expected Zonal checkpoints, validates required reconstruction fields, parses lightweight training-script metadata, and writes a checkpoint inventory to `RESULTS_DIR`.


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''): digest.update(block)
    return digest.hexdigest()

def decode(value):
    return value.decode() if isinstance(value, bytes) else value

def to_builtin(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, bytes):
        return value.decode()
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if hasattr(value, 'detach') and hasattr(value, 'cpu') and hasattr(value, 'tolist'):
        return value.detach().cpu().tolist()
    if hasattr(value, 'tolist') and not isinstance(value, (str, bytes)):
        try:
            return value.tolist()
        except Exception:
            pass
    return value

def infer_geometry_family(name):
    stem = Path(str(name)).name
    if '_dataset' in stem:
        return stem.split('_dataset', 1)[0]
    parts = stem.split('_')
    return parts[0] if parts else stem

def parse_script_metadata(folder):
    scripts = sorted(folder.glob('Training_script*.py'), key=lambda p: (p.name != 'Training_script.py', p.name))
    if not scripts:
        return {'training_scripts': [], 'script_split_seed': None, 'script_eval_fraction': None}
    target = scripts[0]
    text = target.read_text(encoding='utf-8')
    tree = ast.parse(text, filename=str(target))
    names = {'TARGET_NAMES', 'EXTRA_FEAT_COLS', 'INPUT_COLS', 'H5_FILENAME', 'EXPECTED_REPR'}
    values = {}
    for node in tree.body:
        if isinstance(node, ast.Assign):
            targets = [t.id for t in node.targets if isinstance(t, ast.Name)]
            if len(targets) == 1 and targets[0] in names:
                try:
                    values[targets[0]] = ast.literal_eval(node.value)
                except Exception:
                    pass
        elif isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name) and node.target.id in names:
            try:
                values[node.target.id] = ast.literal_eval(node.value)
            except Exception:
                pass
    match = re.search(r'train_test_split\((?P<body>.*?)\)', text, flags=re.S)
    script_eval_fraction, script_split_seed = None, None
    if match:
        body = match.group('body')
        m_frac = re.search(r'test_size\s*=\s*([0-9.]+)', body)
        m_seed = re.search(r'random_state\s*=\s*(\d+)', body)
        if m_frac:
            script_eval_fraction = float(m_frac.group(1))
        if m_seed:
            script_split_seed = int(m_seed.group(1))
    return {
        'training_scripts': [str(p) for p in scripts],
        'script_target_names': to_builtin(values.get('TARGET_NAMES')),
        'script_extra_feat_cols': to_builtin(values.get('EXTRA_FEAT_COLS')),
        'script_input_cols': to_builtin(values.get('INPUT_COLS')),
        'script_h5_filename': values.get('H5_FILENAME'),
        'script_expected_repr': values.get('EXPECTED_REPR'),
        'script_eval_fraction': script_eval_fraction,
        'script_split_seed': script_split_seed,
    }

def target_channels_from_payload(payload):
    for key in ('target_mean', 'target_std', 'target_names'):
        if key in payload:
            arr = np.asarray(to_builtin(payload[key]), dtype=object).reshape(-1)
            if arr.size:
                return int(arr.size)
    arch = to_builtin(payload.get('arch') or {})
    for key in ('out_dim', 'out_channels'):
        if key in arch:
            return int(arch[key])
    return None

def discover():
    rows = []
    required = ['arch', 'target_mean', 'target_std', 'coord_center', 'coord_half_range']
    for ablation_source, families in EXPECTED_FAMILIES.items():
        expected_targets = 2 if ablation_source == JOINT_SOURCE else 1
        for family in families:
            folder = REPO_ROOT / REGIME / ablation_source / family
            script_meta = parse_script_metadata(folder) if folder.exists() else {'training_scripts': [], 'script_split_seed': None, 'script_eval_fraction': None}
            checkpoints = sorted((folder / 'Trained_models').glob('*.pt')) if folder.exists() else []
            if not checkpoints:
                rows.append({
                    'regime': REGIME,
                    'ablation_source': ablation_source,
                    'ablation': ABLATION_LABELS[ablation_source],
                    'model_family': family,
                    'status': 'missing checkpoint',
                    'checkpoint_path': None,
                    'expected_target_channels': expected_targets,
                    **script_meta,
                })
                continue
            for path in checkpoints:
                record = {
                    'regime': REGIME,
                    'ablation_source': ablation_source,
                    'ablation': ABLATION_LABELS[ablation_source],
                    'model_family': family,
                    'checkpoint_path': str(path),
                    'file_size_bytes': int(path.stat().st_size),
                    'sha256': sha256(path),
                    'expected_target_channels': expected_targets,
                    **script_meta,
                }
                try:
                    payload = torch.load(path, map_location='cpu', weights_only=False)
                    missing = [k for k in required if k not in payload]
                    record['missing_keys'] = missing
                    record['arch'] = to_builtin(payload.get('arch'))
                    record['target_names'] = to_builtin(payload.get('target_names'))
                    record['extra_feat_cols'] = to_builtin(payload.get('extra_feat_cols'))
                    record['representation'] = decode(payload.get('representation')) if payload.get('representation') is not None else None
                    record['h5_filename'] = payload.get('h5_filename') or (Path(payload['h5_path']).name if payload.get('h5_path') else None)
                    record['geometry_family'] = payload.get('geometry_family')
                    record['train_sample_ids'] = to_builtin(payload.get('train_sample_ids'))
                    record['val_sample_ids'] = to_builtin(payload.get('val_sample_ids'))
                    record['split_seed'] = to_builtin(payload.get('split_seed'))
                    record['eval_fraction'] = to_builtin(payload.get('eval_fraction'))
                    record['target_channels'] = target_channels_from_payload(payload)
                    record['effective_h5_filename'] = record['h5_filename'] or record.get('script_h5_filename')
                    record['effective_representation'] = record['representation'] or record.get('script_expected_repr')
                    record['effective_geometry_family'] = record['geometry_family'] or infer_geometry_family(record['effective_h5_filename'] or DATASET_PATH.name)
                    if missing:
                        record['status'] = 'incompatible: missing ' + ', '.join(missing)
                    elif record['target_channels'] != expected_targets:
                        record['status'] = f'incompatible: expected {expected_targets} output channel(s), found {record["target_channels"]}'
                    else:
                        record['status'] = 'discovered'
                except Exception as exc:
                    record['status'] = 'incompatible: ' + repr(exc)
                rows.append(record)
    report = pd.DataFrame(rows).sort_values(['ablation_source', 'model_family', 'checkpoint_path'], na_position='last').reset_index(drop=True)
    eh.save_json(report.to_dict(orient='records'), RESULTS_DIR, 'checkpoint_inventory')
    return report

checkpoint_report = discover()
display(checkpoint_report[['ablation_source', 'ablation', 'model_family', 'status', 'checkpoint_path']])


## HDF5 loading, deterministic evaluation split, and fairness checks

This notebook uses one shared Zonal edge dataset and one deterministic 80/20 geometry split for every evaluated model. Fair paired conclusions are restricted to model families that pass the explicit checks below.


In [ ]:
def load_samples(path):
    samples=[]
    with h5py.File(path,'r') as h5:
        representation=decode(h5.attrs.get('representation',''))
        if representation != 'edge': raise ValueError(f'{path.name}: expected representation edge, got {representation!r}')
        for key in sorted(h5['samples'].keys()):
            g=h5['samples'][key]
            def arr(name, default=None): return np.asarray(g[name]) if name in g else default
            coords=arr('node_coords_mm'); stress=arr('stress_max_vm'); life=arr('life_raw')
            if coords is None or stress is None or life is None: raise ValueError(f'{path.name}/{key}: missing required target fields')
            sample_id=decode(g.attrs.get('sample_id', key))
            attrs={str(k):decode(v) for k,v in g.attrs.items()}
            samples.append({'sample_key':key,'sample_id':str(sample_id),'attrs':attrs,'coords':coords.astype('float32'),'stress':stress.astype('float32').reshape(-1),'loglife':np.log10(np.clip(life.astype('float64').reshape(-1),1e-30,None)).astype('float32'),'zone_id':arr('zone_id',np.full(len(coords),-1)).reshape(-1),'subzone_id':arr('subzone_id',np.full(len(coords),np.nan)).reshape(-1),'arc_length_mm':arr('arc_length_mm',np.arange(len(coords),dtype='float32')).reshape(-1),'node_features':arr('node_features',np.empty((len(coords),0),dtype='float32'))})
    return samples

def split_samples(samples):
    rng=np.random.default_rng(SPLIT_SEED); order=rng.permutation(len(samples)); n_eval=max(1,int(round(len(samples)*EVAL_FRACTION)))
    eval_pos=np.sort(order[:n_eval]).tolist(); train_pos=np.sort(order[n_eval:]).tolist()
    return train_pos,eval_pos

def normalize_arch(obj):
    drop = {'out_dim', 'out_channels', 'target_names', 'num_targets'}
    if isinstance(obj, dict):
        return {k: normalize_arch(v) for k, v in sorted(obj.items()) if k not in drop}
    if isinstance(obj, list):
        return [normalize_arch(v) for v in obj]
    return obj

def canonical_json(obj):
    return json.dumps(normalize_arch(to_builtin(obj) or {}), sort_keys=True)

def list_or_empty(value):
    return list(value) if isinstance(value, (list, tuple)) else []

def split_evidence(row, split_record):
    val_ids = row.get('val_sample_ids')
    train_ids = row.get('train_sample_ids')
    if isinstance(val_ids, list) and isinstance(train_ids, list) and val_ids and train_ids:
        return {
            'mode': 'checkpoint_sample_ids',
            'matches_notebook': val_ids == split_record['evaluation_sample_ids'] and train_ids == split_record['training_sample_ids'],
            'signature': hashlib.md5(json.dumps({'train': train_ids, 'val': val_ids}, sort_keys=True).encode('utf-8')).hexdigest(),
        }
    seed = row.get('split_seed', None)
    frac = row.get('eval_fraction', None)
    if seed is None: seed = row.get('script_split_seed', None)
    if frac is None: frac = row.get('script_eval_fraction', None)
    ok = seed == SPLIT_SEED and frac is not None and abs(float(frac) - EVAL_FRACTION) < 1e-12
    return {
        'mode': 'seed_fraction' if seed is not None or frac is not None else 'missing',
        'matches_notebook': bool(ok),
        'signature': f'{seed}|{frac}',
    }

def select_unique_discovered(report, ablation_source, family):
    sub = report[(report['ablation_source'] == ablation_source) & (report['model_family'] == family) & (report['status'] == 'discovered')]
    if len(sub) == 1:
        return sub.iloc[0]
    return None

samples = load_samples(DATASET_PATH)
train_pos, eval_pos = split_samples(samples)
split_record = {
    'regime': REGIME,
    'hdf5_filename': DATASET_PATH.name,
    'geometry_family': infer_geometry_family(DATASET_PATH.name),
    'total_geometry_count': len(samples),
    'evaluation_geometry_count': len(eval_pos),
    'split_seed': SPLIT_SEED,
    'split_fraction': EVAL_FRACTION,
    'training_sample_ids': [samples[i]['sample_id'] for i in train_pos],
    'evaluation_sample_ids': [samples[i]['sample_id'] for i in eval_pos],
    'training_positional_indices': train_pos,
    'evaluation_positional_indices': eval_pos,
    'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'notebook_commit_sha': COMMIT,
    'evaluation_label': EVALUATION_LABEL,
    'independence_basis': 'A new geometry holdout is not proved independent of model validation/checkpoint selection.',
}
eh.save_json(split_record, RESULTS_DIR, 'evaluation_split_provenance')
display(pd.DataFrame([{'regime': REGIME, 'geometries': len(samples), 'evaluation_geometries': len(eval_pos), 'label': EVALUATION_LABEL}]))

fair_rows = []
for family in PAIR_FAMILIES + ['PointNetMLPJoint']:
    joint = select_unique_discovered(checkpoint_report, JOINT_SOURCE, family)
    life = select_unique_discovered(checkpoint_report, LIFE_SOURCE, family)
    row = {'regime': REGIME, 'model_family': family, 'joint_checkpoint': None, 'life_only_checkpoint': None}
    if joint is not None:
        row['joint_checkpoint'] = joint['checkpoint_path']
    if life is not None:
        row['life_only_checkpoint'] = life['checkpoint_path']
    if joint is None or life is None:
        row.update({
            'same_hdf5_file': False if family == 'PointNetMLPJoint' else np.nan,
            'same_geometry_family': False if family == 'PointNetMLPJoint' else np.nan,
            'same_split': False if family == 'PointNetMLPJoint' else np.nan,
            'same_representation': False if family == 'PointNetMLPJoint' else np.nan,
            'same_feature_columns': False if family == 'PointNetMLPJoint' else np.nan,
            'same_arch_except_output': False if family == 'PointNetMLPJoint' else np.nan,
            'target_channels_ok': False,
            'pair_status': 'MISSING COUNTERPART' if family == 'PointNetMLPJoint' else 'NOT FAIRLY COMPARABLE',
            'notes': 'PointNetMLPJoint has no life-only checkpoint under Zonal/Edge_no_stress.' if family == 'PointNetMLPJoint' else 'Missing joint or life-only checkpoint.',
        })
        fair_rows.append(row)
        continue
    joint_split = split_evidence(joint, split_record)
    life_split = split_evidence(life, split_record)
    same_hdf5 = joint.get('effective_h5_filename') == split_record['hdf5_filename'] == life.get('effective_h5_filename')
    same_geom = joint.get('effective_geometry_family') == split_record['geometry_family'] == life.get('effective_geometry_family')
    same_repr = joint.get('effective_representation') == 'edge' == life.get('effective_representation')
    joint_feat = {'extra_feat_cols': list_or_empty(joint.get('extra_feat_cols')), 'input_cols': list_or_empty(joint.get('script_input_cols') or [0, 1])}
    life_feat = {'extra_feat_cols': list_or_empty(life.get('extra_feat_cols')), 'input_cols': list_or_empty(life.get('script_input_cols') or [0, 1])}
    same_feat = joint_feat == life_feat
    same_split = joint_split['matches_notebook'] and life_split['matches_notebook'] and joint_split['signature'] == life_split['signature']
    same_arch = canonical_json(joint.get('arch')) == canonical_json(life.get('arch'))
    target_ok = int(joint.get('target_channels')) == 2 and int(life.get('target_channels')) == 1
    checks = [same_hdf5, same_geom, same_split, same_repr, same_feat, same_arch, target_ok]
    row.update({
        'same_hdf5_file': same_hdf5,
        'same_geometry_family': same_geom,
        'same_split': same_split,
        'same_representation': same_repr,
        'same_feature_columns': same_feat,
        'same_arch_except_output': same_arch,
        'target_channels_ok': target_ok,
        'joint_split_mode': joint_split['mode'],
        'life_split_mode': life_split['mode'],
        'joint_feature_signature': joint_feat,
        'life_feature_signature': life_feat,
        'pair_status': 'FAIRLY COMPARABLE' if all(checks) else 'NOT FAIRLY COMPARABLE',
        'notes': '; '.join([
            f'joint_split={joint_split["mode"]}',
            f'life_split={life_split["mode"]}',
            'same notebook HDF5 / geometry family required',
        ]),
    })
    fair_rows.append(row)

pair_fairness = pd.DataFrame(fair_rows)
eh.save_table(pair_fairness, RESULTS_DIR, 'pair_fairness')
eh.save_json(pair_fairness.to_dict(orient='records'), RESULTS_DIR, 'pair_fairness')
display(pair_fairness[['model_family', 'pair_status', 'same_hdf5_file', 'same_geometry_family', 'same_split', 'same_representation', 'same_feature_columns', 'same_arch_except_output', 'target_channels_ok', 'notes']])


## Model reconstruction and shared-geometry inference

The next cell reconstructs every discovered Zonal checkpoint, evaluates the **same validation-split geometries** for every model, and writes node-level predictions to `RESULTS_DIR`.


In [ ]:
def import_local(path, name):
    spec=importlib.util.spec_from_file_location(name,path); module=importlib.util.module_from_spec(spec); spec.loader.exec_module(module); return module

def reconstruct(row):
    folder=Path(row['checkpoint_path']).parent.parent; family=row['model_family']
    ckpt=torch.load(row['checkpoint_path'],map_location='cpu',weights_only=False); arch=dict(ckpt['arch'])
    pn=import_local(folder/'pn_models.py', f'pn_{folder.parent.name}_{folder.name}_{family}_{row["ablation"]}')
    sys.modules['pn_models']=pn
    if family=='PointNetMLPJoint_FP':
        if not hasattr(pn,'build_fp_model_from_arch'): raise RuntimeError('FP checkpoint requires build_fp_model_from_arch; refusing regular PointNet fallback')
        model=pn.build_fp_model_from_arch(arch)
    elif family in ('PointNetMLPJoint','PointNetMLPJoint_weighted'):
        model=pn.build_model_from_arch(arch)
    else:
        bench=import_local(folder/'benchmarks.py',f'bench_{row.get("ablation","edge")}')
        if not hasattr(bench,'ArGEnTDeepONet'): raise RuntimeError('No ArGEnTDeepONet found in own benchmarks.py')
        defaults={'hidden_dim':128,'num_heads':4,'num_layers':2,'output_dim':128,'out_channels':1,'attention_type':'self','use_sdf':False,'in_ch_geom':2}
        defaults.update({k:v for k,v in arch.items() if k in defaults})
        if 'out_channels' not in arch and 'bias' in ckpt['model_state']: defaults['out_channels']=int(ckpt['model_state']['bias'].shape[0])
        model=bench.ArGEnTDeepONet(**defaults)
    model.load_state_dict(ckpt['model_state'],strict=True)
    model.eval()
    return model,ckpt

def feature_matrix(sample, ckpt):
    cols=ckpt.get('extra_feat_cols',[]) or []; available=sample['node_features']
    if cols and available.shape[1] < len(cols): raise ValueError(f'missing required extra-feature columns: {cols}')
    extra=available[:,:len(cols)] if cols else np.empty((len(sample['coords']),0),dtype='float32')
    center=np.asarray(ckpt['coord_center'],dtype='float32'); half=np.asarray(ckpt['coord_half_range'],dtype='float32'); coords=(sample['coords']-center)/np.maximum(half,1e-8)
    if extra.shape[1]:
        stats=ckpt.get('extra_feat_stats')
        if stats is None: raise ValueError('missing extra-feature normalization statistics')
        if isinstance(stats,dict): mean=np.asarray(stats['mean'],dtype='float32'); std=np.asarray(stats['std'],dtype='float32')
        else: mean=np.asarray(stats[0],dtype='float32'); std=np.asarray(stats[1],dtype='float32')
        extra=(extra-mean)/np.maximum(std,1e-8)
    return coords,extra

def predict(model, sample, ckpt):
    coords,extra=feature_matrix(sample,ckpt); x=torch.from_numpy(coords[None]); q=x.clone()
    with torch.no_grad():
        try: out=model(x,q)
        except TypeError: out=model(torch.cat([x,torch.from_numpy(extra[None])],dim=-1),q)
    out=out.detach().cpu().numpy()
    if out.ndim!=3 or out.shape[0]!=1: raise ValueError(f'prediction shape unexpected: {out.shape}')
    mean=np.asarray(ckpt['target_mean']); std=np.asarray(ckpt['target_std']); out=out*std+mean
    if out.shape[2]==2: return out[0,:,0],out[0,:,1]
    elif out.shape[2]==1: return np.zeros(out.shape[1],dtype='float32'),out[0,:,0]
    raise ValueError(f'unexpected output channels: {out.shape[2]}')

discovered = checkpoint_report[checkpoint_report['status'] == 'discovered'].copy()
ambiguous = discovered.groupby(['ablation_source', 'model_family']).size().reset_index(name='n')
if (ambiguous['n'] > 1).any():
    raise RuntimeError('Expected at most one discovered checkpoint per (ablation_source, model_family).')

node_frames=[]; load_errors=[]
for _,row in discovered.iterrows():
    try:
        model,ckpt=reconstruct(row)
        for i in split_record['evaluation_positional_indices']:
            s=samples[i]
            pred_stress,pred_loglife=predict(model,s,ckpt)
            if int(row['target_channels']) == 1:
                pred_stress = np.full(len(s['coords']), np.nan, dtype='float32')
            base=pd.DataFrame({
                'regime':row['regime'],
                'ablation':row['ablation'],
                'ablation_source':row['ablation_source'],
                'model_family':row['model_family'],
                'target_kind':'joint' if int(row['target_channels']) == 2 else 'life_only',
                'evaluation_label':EVALUATION_LABEL,
                'sample_key':s['sample_key'],
                'sample_id':s['sample_id'],
                'node_idx':np.arange(len(s['coords'])),
                'x_mm':s['coords'][:,0],
                'r_mm':s['coords'][:,1],
                'zone_id':s['zone_id'],
                'subzone_id':s['subzone_id'],
                'arc_length_mm':s['arc_length_mm'],
                'true_stress':s['stress'],
                'pred_stress':pred_stress,
                'true_loglife':s['loglife'],
                'pred_loglife':pred_loglife,
            })
            base['zone_name']=base.zone_id.map(eh.ZONE_ID_TO_NAME)
            base['subzone_name']=base.subzone_id.map(eh.SUBZONE_ID_TO_NAME)
            node_frames.append(base)
    except Exception as exc:
        load_errors.append({'regime':row.regime,'ablation':row.ablation,'model_family':row.model_family,'checkpoint_path':row.checkpoint_path,'status':'load/inference failed: '+repr(exc)})

node_results=pd.concat(node_frames,ignore_index=True) if node_frames else pd.DataFrame()
inference_errors=pd.DataFrame(load_errors)
eh.save_table(node_results, RESULTS_DIR, 'node_results')
eh.save_json(load_errors, RESULTS_DIR, 'inference_errors')
coverage = node_results.groupby(['ablation', 'model_family']).sample_id.nunique().reset_index(name='evaluation_geometries') if not node_results.empty else pd.DataFrame()
if not coverage.empty:
    coverage['expected_evaluation_geometries'] = len(split_record['evaluation_sample_ids'])
    coverage['same_shared_geometry_set'] = coverage['evaluation_geometries'].eq(len(split_record['evaluation_sample_ids']))
    eh.save_table(coverage, RESULTS_DIR, 'evaluation_coverage')
display(inference_errors if not inference_errors.empty else coverage)


## Metrics and paired joint-vs-life-only deltas

Positive paired improvements below mean the **joint** model achieved a **lower absolute LogLife error** than its life-only counterpart.


In [ ]:
pooled_all = eh.pooled_metrics_from_nodes(node_results)
pooled_loglife = pooled_all[pooled_all['target'] == 'LogLife'].copy()
pooled_stress = pooled_all[(pooled_all['target'] == 'Stress') & (pooled_all['ablation'] == 'Edge_joint')].copy()
bins = eh.loglife_bin_metrics(node_results)
zones = eh.zone_metrics_from_nodes(node_results)
geom = eh.geometry_level_metrics(node_results)
geom_summary = eh.geometry_metrics_summary(geom)

for name, frame in [
    ('pooled_metrics_all', pooled_all),
    ('pooled_metrics_loglife', pooled_loglife),
    ('pooled_metrics_stress_joint_only', pooled_stress),
    ('loglife_bin_metrics', bins),
    ('subzone_metrics', zones),
    ('geometry_metrics', geom),
    ('geometry_metrics_summary', geom_summary),
]:
    eh.save_table(frame, RESULTS_DIR, name)

fair_families = pair_fairness.loc[pair_fairness['pair_status'] == 'FAIRLY COMPARABLE', 'model_family'].tolist()

def merge_pair(frame, keys):
    joint = frame[(frame['ablation'] == 'Edge_joint') & (frame['model_family'].isin(fair_families))].copy()
    life = frame[(frame['ablation'] == 'Edge_no_stress') & (frame['model_family'].isin(fair_families))].copy()
    return joint.merge(life, on=keys, suffixes=('_joint', '_life_only'))

paired_pooled = merge_pair(pooled_loglife, ['regime', 'model_family', 'target'])
if not paired_pooled.empty:
    paired_pooled['improvement_MAE'] = paired_pooled['MAE_life_only'] - paired_pooled['MAE_joint']
    paired_pooled['improvement_RMSE'] = paired_pooled['RMSE_life_only'] - paired_pooled['RMSE_joint']
paired_bins = merge_pair(bins, ['regime', 'model_family', 'bin'])
if not paired_bins.empty:
    paired_bins['improvement_MAE'] = paired_bins['MAE_life_only'] - paired_bins['MAE_joint']
    paired_bins['improvement_RMSE'] = paired_bins['RMSE_life_only'] - paired_bins['RMSE_joint']
paired_zones = merge_pair(zones, ['regime', 'model_family', 'subzone_name'])
if not paired_zones.empty:
    paired_zones['improvement_MAE'] = paired_zones['MAE_life_only'] - paired_zones['MAE_joint']
    paired_zones['improvement_RMSE'] = paired_zones['RMSE_life_only'] - paired_zones['RMSE_joint']
paired_geom = merge_pair(geom, ['regime', 'model_family', 'sample_id'])
if not paired_geom.empty:
    paired_geom['joint_abs_min_loglife_error'] = paired_geom['min_loglife_error_decades_joint'].abs()
    paired_geom['life_only_abs_min_loglife_error'] = paired_geom['min_loglife_error_decades_life_only'].abs()
    paired_geom['improvement_abs_min_loglife_error'] = paired_geom['life_only_abs_min_loglife_error'] - paired_geom['joint_abs_min_loglife_error']

for name, frame in [
    ('paired_loglife_overall', paired_pooled),
    ('paired_loglife_bins', paired_bins),
    ('paired_subzone_metrics', paired_zones),
    ('paired_geometry_min_life', paired_geom),
]:
    eh.save_table(frame, RESULTS_DIR, name)

summary_tables = {
    'pair_fairness': pair_fairness[['model_family', 'pair_status', 'notes']].to_dict(orient='records'),
    'fair_families': fair_families,
    'overall_improvement_MAE_mean': float(paired_pooled['improvement_MAE'].mean()) if not paired_pooled.empty else np.nan,
    'overall_improvement_RMSE_mean': float(paired_pooled['improvement_RMSE'].mean()) if not paired_pooled.empty else np.nan,
}
eh.save_json(summary_tables, RESULTS_DIR, 'paired_metric_summary')

display(paired_pooled[['model_family', 'MAE_joint', 'MAE_life_only', 'improvement_MAE', 'RMSE_joint', 'RMSE_life_only', 'improvement_RMSE']] if not paired_pooled.empty else pair_fairness)
display(paired_bins[paired_bins['bin'].isin(['LogLife<4', 'LogLife<3', 'LogLife<2'])][['model_family', 'bin', 'MAE_joint', 'MAE_life_only', 'improvement_MAE', 'RMSE_joint', 'RMSE_life_only', 'improvement_RMSE']] if not paired_bins.empty else pd.DataFrame())
display(paired_zones[paired_zones['subzone_name'].eq('lower_transition')][['model_family', 'subzone_name', 'MAE_joint', 'MAE_life_only', 'improvement_MAE', 'RMSE_joint', 'RMSE_life_only', 'improvement_RMSE']] if not paired_zones.empty else pd.DataFrame())
display(pooled_stress[['model_family', 'MAE', 'RMSE', 'R2 (log)']] if not pooled_stress.empty else pd.DataFrame())


## Figures

Each figure is saved to `RESULTS_DIR/figures` (PNG, with PDF fallback where available).


In [ ]:
def save_fig(fig, name):
    fig.savefig(FIGURES_DIR / f'{name}.png', dpi=150, bbox_inches='tight')
    try:
        fig.savefig(FIGURES_DIR / f'{name}.pdf', bbox_inches='tight')
    except Exception:
        pass

def plot_overall_paired_bars(frame):
    if frame.empty:
        return None
    order = frame['model_family'].tolist()
    x = np.arange(len(order), dtype=float)
    width = 0.34
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True)
    for ax, metric in zip(axes, ['MAE', 'RMSE']):
        ax.bar(x - width / 2, frame[f'{metric}_joint'], width=width, label='Edge_joint')
        ax.bar(x + width / 2, frame[f'{metric}_life_only'], width=width, label='Edge_no_stress')
        ax.set_xticks(x)
        ax.set_xticklabels(order, rotation=15, ha='right')
        ax.set_ylabel(f'LogLife {metric} (decades)')
        ax.set_title(f'Validation-split {metric}')
        for i, imp in enumerate(frame[f'improvement_{metric}']):
            ax.text(x[i], max(frame[f'{metric}_joint'].iloc[i], frame[f'{metric}_life_only'].iloc[i]), f'Δ={imp:+.3f}', ha='center', va='bottom', fontsize=8)
    axes[0].legend(fontsize=8)
    fig.suptitle('Joint stress supervision vs life-only: pooled LogLife metrics')
    fig.tight_layout()
    save_fig(fig, 'paired_loglife_overall_bar')
    return fig

def plot_low_life_bins(frame):
    sub = frame[frame['bin'].isin(['LogLife<4', 'LogLife<3', 'LogLife<2'])].copy()
    if sub.empty:
        return None
    families = sub['model_family'].unique().tolist()
    fig, axes = plt.subplots(2, len(families), figsize=(5.2 * len(families), 7.5), sharex='col')
    if len(families) == 1:
        axes = np.asarray(axes).reshape(2, 1)
    for col, fam in enumerate(families):
        d = sub[sub['model_family'] == fam].set_index('bin').reindex(['LogLife<4', 'LogLife<3', 'LogLife<2'])
        x = np.arange(len(d), dtype=float)
        for row_idx, metric in enumerate(['MAE', 'RMSE']):
            ax = axes[row_idx, col]
            ax.bar(x - 0.18, d[f'{metric}_joint'], width=0.36, label='Edge_joint')
            ax.bar(x + 0.18, d[f'{metric}_life_only'], width=0.36, label='Edge_no_stress')
            ax.set_xticks(x)
            ax.set_xticklabels(d.index, rotation=20)
            ax.set_ylabel(f'LogLife {metric}')
            ax.set_title(fam)
            if row_idx == 0:
                ax.legend(fontsize=8)
    fig.suptitle('Critical low-life node comparison')
    fig.tight_layout()
    save_fig(fig, 'low_life_bins_comparison')
    return fig

def plot_subzone_comparison(frame):
    if frame.empty:
        return None
    order = eh.PRINCIPAL_SUBZONES
    families = frame['model_family'].unique().tolist()
    fig, axes = plt.subplots(len(families), 1, figsize=(13, 4.2 * len(families)), sharex=True)
    if len(families) == 1:
        axes = [axes]
    for ax, fam in zip(axes, families):
        d = frame[frame['model_family'] == fam].set_index('subzone_name').reindex(order)
        x = np.arange(len(order), dtype=float)
        ax.bar(x - 0.18, d['MAE_joint'], width=0.36, label='Edge_joint')
        ax.bar(x + 0.18, d['MAE_life_only'], width=0.36, label='Edge_no_stress')
        ax.set_ylabel('LogLife MAE')
        ax.set_title(f'{fam} subzone comparison')
        ax.legend(fontsize=8)
        if 'lower_transition' in d.index.tolist():
            lt = d.index.tolist().index('lower_transition')
            ax.axvspan(lt - 0.5, lt + 0.5, color='gold', alpha=0.12)
        ax.set_xticks(x)
        ax.set_xticklabels(order, rotation=30, ha='right')
    fig.suptitle('Subzone LogLife MAE comparison')
    fig.tight_layout()
    save_fig(fig, 'subzone_metrics_comparison')
    return fig

def plot_geometry_pair_scatter(frame):
    if frame.empty:
        return None
    fig, ax = plt.subplots(figsize=(6.2, 6.2))
    for fam, g in frame.groupby('model_family'):
        ax.scatter(g['life_only_abs_min_loglife_error'], g['joint_abs_min_loglife_error'], s=16, alpha=0.65, label=fam)
    lim = max(frame['life_only_abs_min_loglife_error'].max(), frame['joint_abs_min_loglife_error'].max())
    ax.plot([0, lim], [0, lim], 'k--', lw=1)
    ax.set_xlabel('Life-only |min LogLife error| (decades)')
    ax.set_ylabel('Joint |min LogLife error| (decades)')
    ax.set_title('Per-geometry minimum-life error scatter')
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_fig(fig, 'per_geometry_min_life_scatter')
    return fig

def plot_joint_stress(frame):
    if frame.empty:
        return None
    d = frame.set_index('model_family').loc[:, ['MAE', 'RMSE']]
    x = np.arange(len(d), dtype=float)
    width = 0.34
    fig, ax = plt.subplots(figsize=(8.5, 4.5))
    ax.bar(x - width / 2, d['MAE'], width=width, label='Stress MAE')
    ax.bar(x + width / 2, d['RMSE'], width=width, label='Stress RMSE')
    ax.set_xticks(x)
    ax.set_xticklabels(d.index, rotation=15, ha='right')
    ax.set_ylabel('Stress error (MPa)')
    ax.set_title('Joint-model stress performance (life-only models do not output stress)')
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_fig(fig, 'joint_model_stress_error')
    return fig

_ = plot_overall_paired_bars(paired_pooled)
_ = plot_low_life_bins(paired_bins)
_ = plot_subzone_comparison(paired_zones[paired_zones['status_joint'].eq('ok') & paired_zones['status_life_only'].eq('ok')] if not paired_zones.empty else paired_zones)
_ = plot_geometry_pair_scatter(paired_geom)
_ = plot_joint_stress(pooled_stress)
plt.show()


## Cautious conclusion

The final cell writes a short report answering whether auxiliary stress supervision helps fatigue-life prediction overall, in critical low-life nodes, and specifically in the lower transition.


In [ ]:
def directional_statement(values, tol=1e-3):
    vals = pd.Series(values, dtype='float64').dropna()
    if vals.empty:
        return {'label': 'insufficient evidence', 'median': np.nan, 'fraction_positive': np.nan, 'n': 0}
    median = float(vals.median())
    frac_positive = float((vals > 0).mean())
    if median > tol and frac_positive >= 0.60:
        label = 'suggests improvement from joint stress supervision'
    elif median < -tol and frac_positive <= 0.40:
        label = 'suggests no overall improvement from joint stress supervision'
    else:
        label = 'is mixed / model-dependent'
    return {'label': label, 'median': median, 'fraction_positive': frac_positive, 'n': int(len(vals))}

overall_mae = directional_statement(paired_pooled['improvement_MAE'] if not paired_pooled.empty else [])
overall_rmse = directional_statement(paired_pooled['improvement_RMSE'] if not paired_pooled.empty else [])
low_life_focus = paired_bins[paired_bins['bin'].isin(['LogLife<3', 'LogLife<2'])] if not paired_bins.empty else pd.DataFrame()
low_life_mae = directional_statement(low_life_focus['improvement_MAE'] if not low_life_focus.empty else [])
lower_transition = paired_zones[paired_zones['subzone_name'].eq('lower_transition')] if not paired_zones.empty else pd.DataFrame()
lower_transition_mae = directional_statement(lower_transition['improvement_MAE'] if not lower_transition.empty else [])

stress_lines = []
if pooled_stress.empty:
    stress_lines.append('- No joint-model stress metrics were produced.')
else:
    for _, row in pooled_stress.sort_values('model_family').iterrows():
        stress_lines.append(f"- `{row['model_family']}` stress MAE={row['MAE']:.4f} MPa, RMSE={row['RMSE']:.4f} MPa.")

summary_parts = [
    f'# Joint stress supervision ablation summary\n\nRepository commit: `{COMMIT}`\n\nEvaluation label: **{EVALUATION_LABEL}**. The split is a deterministic 20% geometry holdout (seed 42); independence from checkpoint selection is not proved.',
    '## Fairness and scope',
    pair_fairness[['model_family', 'pair_status', 'notes']].to_markdown(index=False),
    '## Answers',
    f"- **Overall pooled fatigue-life accuracy:** {overall_mae['label']} (paired LogLife MAE median improvement `{overall_mae['median']:+.4f}` decades; positive means joint is better). RMSE evidence {overall_rmse['label']} with median `{overall_rmse['median']:+.4f}` decades.",
    f"- **Critical low-life nodes (`LogLife<3` and `<2`):** {low_life_mae['label']} (median paired MAE improvement `{low_life_mae['median']:+.4f}` decades across `{low_life_mae['n']}` paired bin records).",
    f"- **Lower transition subzone:** {lower_transition_mae['label']} (median paired MAE improvement `{lower_transition_mae['median']:+.4f}` decades across `{lower_transition_mae['n']}` paired subzone records).",
    '- **PointNetMLPJoint paired conclusion:** excluded because `Zonal/Edge_no_stress/PointNetMLPJoint` is missing.',
    '## Joint-model stress performance',
]
summary_parts.extend(stress_lines)
summary = '\n'.join(summary_parts)

summary_payload = {
    'commit': COMMIT,
    'evaluation_label': EVALUATION_LABEL,
    'fair_families': fair_families,
    'overall_mae': overall_mae,
    'overall_rmse': overall_rmse,
    'low_life_mae': low_life_mae,
    'lower_transition_mae': lower_transition_mae,
    'pointnet_joint_status': 'missing life-only counterpart',
}
(RESULTS_DIR / 'joint_stress_supervision_summary.md').write_text(summary, encoding='utf-8')
eh.save_json(summary_payload, RESULTS_DIR, 'joint_stress_supervision_summary')
display(Markdown(summary))
